In [22]:
import pandas as pd
import numpy as np

df_train = pd.read_csv('../Raw-Dataset/train.csv')
df_test = pd.read_csv('../Raw-Dataset/test.csv')



In [23]:
m_train = df_train.shape[0]
m_test = df_test.shape[0]
train_size = m_train / (m_test + m_train)
train_size

0.400000537624292

In [24]:
df_train['dataset'] = 'train'
df_test['dataset'] = 'test'
df_test['target'] = 'None'

In [25]:
df = pd.concat([df_train, df_test], axis=0).reset_index(drop=True)
df.shape

(1488028, 60)

- Seperate columns as **categorical**, **binary** and **numeric or ordinal**
- Seperate Category columns with need **One-Hot encoding** or **Target encoding**

How to preprocess?
#### Missing Values
- Replace `-1` with NaN.
- drop columns (`ps_car_03_cat`, `ps_car_05_cat`) with high missing values
- consider missing in (`ps_car_07_cat`) as **new category**
- Impute missing values in other category columns with mode.
- Impute with median for numeric columns with missing values.
- Classify columns based on observation in EDA.
#### Encoding 
- Classify hihg cardinality cat columns for Target encoding.
- low cardinality ones for OneHot encoding.

#### handling Outliers
- Don't do aggresive Outlier removal.
- Consider only for extreme outliers.

Classify columns:
- highly_dominant_features = `ps_car_02_cat`(83%), `ps_car_08_cat`(83%), `ps_car_07_cat`(94%) (consider removal later training)
- Extreme_dominant_features = `ps_car_10_cat` (99%), `ps_ind_14` (99%). (remove them, less informative)

- cat_Target_encoding = `ps_car_11_cat`
- cat_OneHot = remaining cat features

No Correlations.

HIgh Class imbalance use `precision_recall` or `AUC` while cv.

In [26]:
df.replace(-1, np.nan, inplace=True)

,id,target,ps_ind_01,ps_ind_02_cat,ps_ind_03,ps_ind_04_cat,ps_ind_05_cat,ps_ind_06_bin,ps_ind_07_bin,ps_ind_08_bin,...,ps_calc_12,ps_calc_13,ps_calc_14,ps_calc_15_bin,ps_calc_16_bin,ps_calc_17_bin,ps_calc_18_bin,ps_calc_19_bin,ps_calc_20_bin,dataset
0,7,0,2,2.0,5,1.0,0.0,0,1,0,...,1,5,8,0,1,1,0,0,1,train
1,9,0,1,1.0,7,0.0,0.0,0,0,1,...,1,1,9,0,1,1,0,1,0,train
2,13,0,5,4.0,9,1.0,0.0,0,0,1,...,2,7,7,0,1,1,0,1,0,train
3,16,0,0,1.0,2,0.0,0.0,1,0,0,...,2,4,9,0,0,0,0,0,0,train
4,17,0,0,2.0,0,1.0,0.0,1,0,0,...,1,1,3,0,0,0,1,1,0,train
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1488023,1488022,None,0,1.0,6,0.0,0.0,0,1,0,...,2,3,4,0,1,0,0,1,0,test
1488024,1488023,None,5,3.0,5,1.0,0.0,0,0,1,...,2,2,11,0,0,1,1,0,0,test
1488025,1488024,None,0,1.0,5,0.0,0.0,1,0,0,...,2,2,11,0,1,1,0,0,0,test
1488026,1488025,None,6,1.0,5,1.0,0.0,0,0,0,...,1,2,7,1,1,0,0,0,0,test


In [27]:
# Sepearte features
columns = df.columns
cat_cols = [col for col in columns if col.endswith('_cat')]
bin_cols = [col for col in columns if col.endswith('_bin')]
numeric_cols = [ col for col in columns if col not in cat_cols + bin_cols + ['id', 'target']]

# Category columns
cat_cols_remove = ['ps_car_03_cat', 'ps_car_05_cat']
cat_cols_impute = ['ps_ind_02_cat', 'ps_ind_04_cat', 'ps_car_01_cat', 'ps_car_02_cat', 'ps_car_09_cat', 'ps_ind_05_cat']

num_cols_impute = ['ps_car_11', 'ps_car_12']
num_cols_moderate_missing = ['ps_reg_03', 'ps_car_14']

cat_cols_highly_dominant = ['ps_car_02_cat', 'ps_car_08_cat', 'ps_car_07_cat']
cat_cols_Extreme_dominant = ['ps_car_10_cat', 'ps_ind_14']


## Handle Missing

- (`ps_car_03_cat`, `ps_car_05_cat`) drop them

In [28]:
df.drop(columns=cat_cols_remove, inplace=True)

- consider missing in (`ps_car_07_cat`) as **new category**
- Impute missing values in other category columns with mode.
- Impute with median for numeric columns with missing values.

In [29]:
df['ps_car_07_cat'] = df['ps_car_07_cat'].replace(np.nan, -1, inplace=True)

# cat_cols_impute = ['ps_ind_02_cat', 'ps_ind_04_cat', 'ps_car_01_cat', 'ps_car_02_cat', 'ps_car_09_cat', 'ps_ind_05_cat']
cat_impute_dict = {}

for col in cat_cols_impute:
    cat_impute_dict[col] = df[col].mode().iloc[0]

df[cat_cols_impute] = df[cat_cols_impute].fillna(cat_impute_dict)

# num_cols_impute = ['ps_car_11', 'ps_car_12']
# num_cols_moderate_missing = ['ps_reg_03', 'ps_car_14']

num_low_impute_dict = {}
num_mod_impute_dict = {}

for col in num_cols_impute:
    num_low_impute_dict[col] = df[col].median()

for col in num_cols_moderate_missing:
    num_mod_impute_dict[col] = df[col].median()

df[num_cols_impute] = df[num_cols_impute].fillna(num_low_impute_dict)
df[num_cols_moderate_missing] = df[num_cols_moderate_missing].fillna(num_mod_impute_dict)



C:\Users\Adarsh\AppData\Local\Temp\ipykernel_12256\3040550082.py:1: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment using an inplace method.
Such inplace method never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy (due to Copy-on-Write).

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' instead, to perform the operation inplace on the original object, or try to avoid an inplace operation using 'df[col] = df[col].method(value)'.

See the documentation for a more detailed explanation: https://pandas.pydata.org/pandas-docs/stable/user_guide/copy_on_write.html
  df['ps_car_07_cat'] = df['ps_car_07_cat'].replace(np.nan, -1, inplace=True)


#### Remove cat cols with Domainant Category

In [30]:
# cat_cols_highly_dominant = ['ps_car_02_cat', 'ps_car_08_cat', 'ps_car_07_cat']
# cat_cols_Extreme_dominant = ['ps_car_10_cat', 'ps_ind_14'] # drop them

df.drop(columns=cat_cols_Extreme_dominant, inplace=True)


In [31]:
df_train = df[df['dataset'] == 'train']
df_test = df[df['dataset'] == 'test']

df_train.drop(columns=['dataset']).to_csv('../CLeaned-Dataset/train.csv', index=False)
df_test.drop(columns=['target', 'dataset']).to_csv('../Cleaned-Dataset/test.csv', index=False)
